In [ ]:
import tkinter as tk
from tkinter import messagebox, ttk
from tkinter.scrolledtext import ScrolledText
import joblib
import re
import string
import nltk
import csv
import os
from datetime import datetime
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

try:
    model = joblib.load("multi_best_lightgbm_model.pkl")
    tfidf = joblib.load("multi_tfidf.pkl")
except Exception as e:
    raise SystemExit(f"Error loading model files:\n{e}")

CLASS_NAMES = {
    0: "Hate Speech",
    1: "Offensive Language",
    2: "Neutral Content"
}

EVALUATION_FILE = "human_evaluation_results.csv"


def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    words = [
        w for w in text.split()
        if w not in stop_words
    ]

    words = [
        stemmer.stem(w)
        for w in words
    ]

    return " ".join(words)


root = tk.Tk()

root.title(
    "Prototype for Multi Class Classification and Human Evaluation"
)

root.geometry("1100x800")
root.minsize(900, 650)
root.configure(bg="white")
root.resizable(True, True)

try:
    root.state("zoomed")
except:
    pass


tk.Label(
    root,
    text="Prototype for Multi Class Classification and Human Evaluation",
    font=("Arial", 22, "bold"),
    bg="white"
).pack(pady=20)


tk.Label(
    root,
    text="Enter Text",
    font=("Arial", 14, "bold"),
    bg="white"
).pack(
    anchor="w",
    padx=40
)


text_box = ScrolledText(
    root,
    height=10,
    font=("Arial", 12),
    wrap=tk.WORD
)

text_box.pack(
    fill="x",
    padx=40,
    pady=10
)


result_frame = tk.Frame(
    root,
    bg="white"
)

result_frame.pack(
    fill="x",
    padx=40,
    pady=15
)


tk.Label(
    result_frame,
    text="Prediction Result",
    font=("Arial", 15, "bold"),
    bg="white"
).pack(
    anchor="w",
    pady=(0, 10)
)


prediction_label = tk.Label(
    result_frame,
    text="Predicted Class : -",
    font=("Arial", 14),
    bg="white"
)

prediction_label.pack(
    anchor="w"
)


confidence_label = tk.Label(
    result_frame,
    text="Confidence : -",
    font=("Arial", 14),
    bg="white"
)

confidence_label.pack(
    anchor="w",
    pady=5
)


status_bar = tk.Label(
    root,
    text="Status : Ready",
    bg="#f2f2f2",
    anchor="w",
    padx=10
)

status_bar.pack(
    side="bottom",
    fill="x"
)


current_prediction = None
current_confidence = None
current_text = ""


def predict_text():

    global current_prediction
    global current_confidence
    global current_text

    user_text = text_box.get(
        "1.0",
        tk.END
    ).strip()

    if not user_text:
        messagebox.showwarning(
            "Warning",
            "Please enter some text."
        )
        return

    processed = preprocess_text(
        user_text
    )

    try:
        vector = tfidf.transform(
            [processed]
        )

        prediction = int(
            model.predict(vector)[0]
        )

        probs = model.predict_proba(
            vector
        )[0]

        confidence = (
            probs[prediction] * 100
        )

        current_prediction = prediction
        current_confidence = confidence
        current_text = user_text

        prediction_label.config(
            text=(
                f"Predicted Class : "
                f"{CLASS_NAMES[prediction]}"
            )
        )

        confidence_label.config(
            text=(
                f"Confidence : "
                f"{confidence:.2f}%"
            )
        )

        if prediction == 0:
            prediction_label.config(
                fg="red"
            )
        elif prediction == 1:
            prediction_label.config(
                fg="orange"
            )
        else:
            prediction_label.config(
                fg="green"
            )

        status_bar.config(
            text="Status : Prediction completed successfully."
        )

    except Exception as e:
        messagebox.showerror(
            "Prediction Error",
            str(e)
        )


def clear_text():

    global current_prediction
    global current_confidence
    global current_text

    text_box.delete(
        "1.0",
        tk.END
    )

    prediction_label.config(
        text="Predicted Class : -",
        fg="black"
    )

    confidence_label.config(
        text="Confidence : -"
    )

    current_prediction = None
    current_confidence = None
    current_text = ""

    status_bar.config(
        text="Status : Ready"
    )


def open_human_evaluation():

    if current_prediction is None:
        messagebox.showwarning(
            "Human Evaluation",
            "Please make a prediction before starting the human evaluation."
        )
        return

    evaluation_window = tk.Toplevel(
        root
    )

    evaluation_window.title(
        "Human Evaluation"
    )

    evaluation_window.geometry(
        "900x750"
    )

    evaluation_window.minsize(
        700,
        600
    )

    evaluation_window.configure(
        bg="white"
    )

    evaluation_window.resizable(
        True,
        True
    )

    canvas = tk.Canvas(
        evaluation_window,
        bg="white",
        highlightthickness=0
    )

    scrollbar = ttk.Scrollbar(
        evaluation_window,
        orient="vertical",
        command=canvas.yview
    )

    scrollable_frame = tk.Frame(
        canvas,
        bg="white"
    )

    scrollable_frame.bind(
        "<Configure>",
        lambda event: canvas.configure(
            scrollregion=canvas.bbox("all")
        )
    )

    canvas_window = canvas.create_window(
        (0, 0),
        window=scrollable_frame,
        anchor="nw"
    )

    def resize_scroll_frame(event):
        canvas.itemconfig(
            canvas_window,
            width=event.width
        )

    canvas.bind(
        "<Configure>",
        resize_scroll_frame
    )

    canvas.configure(
        yscrollcommand=scrollbar.set
    )

    canvas.pack(
        side="left",
        fill="both",
        expand=True
    )

    scrollbar.pack(
        side="right",
        fill="y"
    )

    def mousewheel(event):
        canvas.yview_scroll(
            int(-1 * (event.delta / 120)),
            "units"
        )

    canvas.bind(
        "<MouseWheel>",
        mousewheel
    )

    content = tk.Frame(
        scrollable_frame,
        bg="white"
    )

    content.pack(
        fill="both",
        expand=True,
        padx=45,
        pady=25
    )

    tk.Label(
        content,
        text="Human Evaluation",
        font=("Arial", 22, "bold"),
        bg="white"
    ).pack(
        pady=(0, 8)
    )

    tk.Label(
        content,
        text=(
            "Please review the model prediction and provide your assessment."
        ),
        font=("Arial", 11),
        bg="white"
    ).pack(
        pady=(0, 25)
    )

    tk.Label(
        content,
        text="Text Being Evaluated",
        font=("Arial", 14, "bold"),
        bg="white"
    ).pack(
        anchor="w"
    )

    evaluation_text = ScrolledText(
        content,
        height=7,
        font=("Arial", 11),
        wrap=tk.WORD
    )

    evaluation_text.pack(
        fill="x",
        pady=(8, 20)
    )

    evaluation_text.insert(
        "1.0",
        current_text
    )

    evaluation_text.config(
        state="disabled"
    )

    tk.Label(
        content,
        text="Model Prediction",
        font=("Arial", 14, "bold"),
        bg="white"
    ).pack(
        anchor="w"
    )

    tk.Label(
        content,
        text=(
            f"Predicted Class : "
            f"{CLASS_NAMES[current_prediction]}"
        ),
        font=("Arial", 12),
        bg="white"
    ).pack(
        anchor="w",
        pady=(8, 2)
    )

    tk.Label(
        content,
        text=(
            f"Confidence : "
            f"{current_confidence:.2f}%"
        ),
        font=("Arial", 12),
        bg="white"
    ).pack(
        anchor="w",
        pady=(0, 20)
    )

    tk.Label(
        content,
        text="Prediction Assessment",
        font=("Arial", 14, "bold"),
        bg="white"
    ).pack(
        anchor="w"
    )

    tk.Label(
        content,
        text="Is the model prediction correct?",
        font=("Arial", 11, "bold"),
        bg="white"
    ).pack(
        anchor="w",
        pady=(8, 5)
    )

    correctness_var = tk.StringVar(
        value=""
    )

    tk.Radiobutton(
        content,
        text="Prediction Correct",
        variable=correctness_var,
        value="Correct",
        font=("Arial", 11),
        bg="white"
    ).pack(
        anchor="w",
        pady=2
    )

    tk.Radiobutton(
        content,
        text="Prediction Incorrect",
        variable=correctness_var,
        value="Incorrect",
        font=("Arial", 11),
        bg="white"
    ).pack(
        anchor="w",
        pady=2
    )

    tk.Label(
        content,
        text="Evaluation Factors",
        font=("Arial", 14, "bold"),
        bg="white"
    ).pack(
        anchor="w",
        pady=(25, 5)
    )

    tk.Label(
        content,
        text=(
            "Please rate each factor from 1 to 5, where "
            "1 = Very Poor and 5 = Very Good."
        ),
        font=("Arial", 10),
        bg="white"
    ).pack(
        anchor="w",
        pady=(0, 15)
    )

    tk.Label(
        content,
        text="1. Prediction Accuracy",
        font=("Arial", 11, "bold"),
        bg="white"
    ).pack(
        anchor="w"
    )

    accuracy_var = tk.StringVar(
        value=""
    )

    accuracy_combo = ttk.Combobox(
        content,
        textvariable=accuracy_var,
        values=["1", "2", "3", "4", "5"],
        state="readonly",
        width=15
    )

    accuracy_combo.pack(
        anchor="w",
        pady=(5, 15)
    )

    tk.Label(
        content,
        text="2. Ease of Use",
        font=("Arial", 11, "bold"),
        bg="white"
    ).pack(
        anchor="w"
    )

    usability_var = tk.StringVar(
        value=""
    )

    usability_combo = ttk.Combobox(
        content,
        textvariable=usability_var,
        values=["1", "2", "3", "4", "5"],
        state="readonly",
        width=15
    )

    usability_combo.pack(
        anchor="w",
        pady=(5, 15)
    )

    tk.Label(
        content,
        text="3. Clarity of Result",
        font=("Arial", 11, "bold"),
        bg="white"
    ).pack(
        anchor="w"
    )

    clarity_var = tk.StringVar(
        value=""
    )

    clarity_combo = ttk.Combobox(
        content,
        textvariable=clarity_var,
        values=["1", "2", "3", "4", "5"],
        state="readonly",
        width=15
    )

    clarity_combo.pack(
        anchor="w",
        pady=(5, 15)
    )

    tk.Label(
        content,
        text="4. Confidence in the Result",
        font=("Arial", 11, "bold"),
        bg="white"
    ).pack(
        anchor="w"
    )

    confidence_factor_var = tk.StringVar(
        value=""
    )

    confidence_factor_combo = ttk.Combobox(
        content,
        textvariable=confidence_factor_var,
        values=["1", "2", "3", "4", "5"],
        state="readonly",
        width=15
    )

    confidence_factor_combo.pack(
        anchor="w",
        pady=(5, 25)
    )

    tk.Label(
        content,
        text="Open Comments",
        font=("Arial", 14, "bold"),
        bg="white"
    ).pack(
        anchor="w"
    )

    tk.Label(
        content,
        text=(
            "Please provide any comments about the prediction "
            "or the prototype."
        ),
        font=("Arial", 10),
        bg="white"
    ).pack(
        anchor="w",
        pady=(5, 8)
    )

    comments_box = ScrolledText(
        content,
        height=7,
        font=("Arial", 11),
        wrap=tk.WORD
    )

    comments_box.pack(
        fill="x",
        pady=(0, 25)
    )

    tk.Label(
        content,
        text="Overall Feedback",
        font=("Arial", 14, "bold"),
        bg="white"
    ).pack(
        anchor="w"
    )

    tk.Label(
        content,
        text=(
            "Please provide your overall feedback about "
            "the prototype."
        ),
        font=("Arial", 10),
        bg="white"
    ).pack(
        anchor="w",
        pady=(5, 8)
    )

    overall_box = ScrolledText(
        content,
        height=7,
        font=("Arial", 11),
        wrap=tk.WORD
    )

    overall_box.pack(
        fill="x",
        pady=(0, 25)
    )

    def save_evaluation():

        correctness = correctness_var.get()
        accuracy = accuracy_var.get()
        usability = usability_var.get()
        clarity = clarity_var.get()
        confidence_factor = confidence_factor_var.get()

        comments = comments_box.get(
            "1.0",
            tk.END
        ).strip()

        overall_feedback = overall_box.get(
            "1.0",
            tk.END
        ).strip()

        if not correctness:
            messagebox.showwarning(
                "Incomplete Evaluation",
                "Please select Prediction Correct or Prediction Incorrect."
            )
            return

        if not accuracy:
            messagebox.showwarning(
                "Incomplete Evaluation",
                "Please rate Prediction Accuracy."
            )
            return

        if not usability:
            messagebox.showwarning(
                "Incomplete Evaluation",
                "Please rate Ease of Use."
            )
            return

        if not clarity:
            messagebox.showwarning(
                "Incomplete Evaluation",
                "Please rate Clarity of Result."
            )
            return

        if not confidence_factor:
            messagebox.showwarning(
                "Incomplete Evaluation",
                "Please rate Confidence in the Result."
            )
            return

        if not overall_feedback:
            messagebox.showwarning(
                "Incomplete Evaluation",
                "Please provide Overall Feedback."
            )
            return

        file_exists = os.path.exists(
            EVALUATION_FILE
        )

        try:

            with open(
                EVALUATION_FILE,
                "a",
                newline="",
                encoding="utf-8"
            ) as file:

                writer = csv.writer(file)

                if not file_exists:

                    writer.writerow([
                        "Timestamp",
                        "Text",
                        "Model Prediction",
                        "Model Confidence",
                        "Prediction Assessment",
                        "Prediction Accuracy",
                        "Ease of Use",
                        "Clarity of Result",
                        "Confidence in Result",
                        "Open Comments",
                        "Overall Feedback"
                    ])

                writer.writerow([
                    datetime.now().strftime(
                        "%Y-%m-%d %H:%M:%S"
                    ),
                    current_text,
                    CLASS_NAMES[current_prediction],
                    f"{current_confidence:.2f}%",
                    correctness,
                    accuracy,
                    usability,
                    clarity,
                    confidence_factor,
                    comments,
                    overall_feedback
                ])

            messagebox.showinfo(
                "Evaluation Saved",
                "Human evaluation has been saved successfully."
            )

            evaluation_window.destroy()

            status_bar.config(
                text="Status : Human evaluation saved successfully."
            )

        except Exception as e:

            messagebox.showerror(
                "Save Error",
                str(e)
            )

    button_frame = tk.Frame(
        content,
        bg="white"
    )

    button_frame.pack(
        pady=(0, 40)
    )

    tk.Button(
        button_frame,
        text="Save Evaluation",
        width=20,
        font=("Arial", 11, "bold"),
        bg="#1976D2",
        fg="white",
        command=save_evaluation
    ).grid(
        row=0,
        column=0,
        padx=10
    )

    tk.Button(
        button_frame,
        text="Close",
        width=20,
        font=("Arial", 11, "bold"),
        command=evaluation_window.destroy
    ).grid(
        row=0,
        column=1,
        padx=10
    )

    canvas.yview_moveto(0)


def exit_app():
    root.destroy()


button_frame = tk.Frame(
    root,
    bg="white"
)

button_frame.pack(
    pady=20
)


tk.Button(
    button_frame,
    text="Predict",
    width=15,
    font=("Arial", 12, "bold"),
    bg="#1976D2",
    fg="white",
    command=predict_text
).grid(
    row=0,
    column=0,
    padx=10
)


tk.Button(
    button_frame,
    text="Human Evaluation",
    width=20,
    font=("Arial", 12, "bold"),
    bg="#6A1B9A",
    fg="white",
    command=open_human_evaluation
).grid(
    row=0,
    column=1,
    padx=10
)


tk.Button(
    button_frame,
    text="Clear",
    width=15,
    font=("Arial", 12, "bold"),
    bg="#F9A825",
    command=clear_text
).grid(
    row=0,
    column=2,
    padx=10
)


tk.Button(
    button_frame,
    text="Exit",
    width=15,
    font=("Arial", 12, "bold"),
    bg="#D32F2F",
    fg="white",
    command=exit_app
).grid(
    row=0,
    column=3,
    padx=10
)


root.mainloop()